# Training the Fraud Detection model with Ray by using Codeflare

The example fraud detection model is very small and quickly trained. However, for many large models, training requires multiple GPUs and often multiple machines. In this notebook, you learn how to train a model by using Ray on OpenShift AI to scale out the model training. You use the Codeflare SDK to create the cluster and submit the job. You can find detailed documentation for the SDK [here](https://project-codeflare.github.io/codeflare-sdk/detailed-documentation/).

For this procedure, you need to use codeflare-sdk 0.19.1 (or later).  Begin by installing the SDK if it's not already installed or up to date:

In [1]:
!pip install --upgrade codeflare-sdk>=0.19.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp 2.14.6 requires kubernetes<31,>=8.0.0, but you have kubernetes 35.0.0 which is incompatible.


### Preparing the data

Normally, the training data for your model would be available in a shared location. For this example, the data is local. You must upload it to your object storage so that you can see how data loading from a shared data source works. After you upload the data, you can work with it by using Ray Data so that it is properly shared across the worker machines.

In [2]:
import sys
sys.path.append('./utils')

import utils.s3

utils.s3.upload_directory_to_s3("data", "data")
print("---")
utils.s3.list_objects("data")

data/test.csv -> data/test.csv
data/train.csv -> data/train.csv
data/validate.csv -> data/validate.csv
---
data/test.csv
data/train.csv
data/validate.csv


### Authenticate to the cluster by using the OpenShift console login

You must create the Kubernetes objects for Ray Clusters using the Codeflare SDK. In order to do so, you need access permission for your own namespace. The easiest way to set up access is by using the OpenShift CLI `oc` client. 

From the OpenShift web console, you can generate an `oc login` command that includes your token and server information. You can use the command to log in to the OpenShift CLI. 

1. To generate the command, select **Copy login command** from the username drop-down menu at the top right of the web console.

    <figure>
        <img src="./assets/copy-login.png"  alt="copy login"  >
    <figure/>

2. Click **Display token**.

3. Below **Log in with this token**, take note of the parameters for token and server.
   For example:
    ```
    oc login --token=sha256~LongString --server=https://api.your-cluster.domain.com:6443
    ```    
    - token: `sha256~LongString`
    - server: `https://api.your-cluster.domain.com:6443`
    
4. In the following code cell, in the TokenAuthentication object, replace the token and server values with the values that you noted in Step 3.
   For example:
   ```
   auth = TokenAuthentication(
       token = "sha256~LongString",
       server = "https://api.your-cluster.domain.com:6443",
       skip_tls=False
   )
   auth.login()
   ```


In [5]:
from codeflare_sdk import TokenAuthentication
# Create authentication object for user permissions
# IF unused, SDK will automatically check for default kubeconfig, then in-cluster config
# KubeConfigFileAuthentication can also be used to specify kubeconfig path manually
# oc login --token=sha256~w2CCqpKiPeWUzeQBAbLOHLF2X-Ou3p9La596EGtIJ6s --server=https://api.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com:6443
auth = TokenAuthentication(
    token = "sha256~w2CCqpKiPeWUzeQBAbLOHLF2X-Ou3p9La596EGtIJ6s",
    server = "https://api.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com:6443",
    skip_tls=True
)
auth.login()

Insecure request warnings have been disabled


/tmp/ipykernel_374/3587113012.py:6: DeprecationWarning: Use kube_authkit.AuthConfig with token strategy instead.
  auth = TokenAuthentication(
/tmp/ipykernel_374/3587113012.py:6: DeprecationWarning: TokenAuthentication is deprecated and will be removed in a future version. Please use kube-authkit's AuthConfig directly. See: https://github.com/opendatahub-io/kube-authkit
  auth = TokenAuthentication(


'Logged into https://api.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com:6443'

## Create a Ray cluster

### Configure a Ray cluster

CodeFlare allows you to specify parameters, such as number of workers, image, and kueue local queue name.  A full list of parameters is available [here](https://project-codeflare.github.io/codeflare-sdk/detailed-documentation/cluster/config.html).

In [11]:
from codeflare_sdk import Cluster, ClusterConfiguration

cluster = Cluster(ClusterConfiguration(
    name="raycluster-cpu",
    # head_extended_resource_requests={'nvidia.com/gpu': 0}, # Commented out as we don't need GPUs for this example
    # worker_extended_resource_requests={'nvidia.com/gpu': 0},
    num_workers=2,
    worker_cpu_requests=1,
    worker_cpu_limits=4,
    worker_memory_requests=2,
    worker_memory_limits=4
))


Yaml resources loaded for raycluster-cpu


Output()

### Start the cluster

If you have a running cluster that you want to connect to, skip to the next cell.

To start a cluster, run the following cell to create the necessary Kubernetes objects to run the Ray cluster. This step might take a few minutes to complete.

In [12]:
cluster.up()
cluster.wait_ready()

A conflict occurred with the RayCluster resource.
Only one RayCluster with the same name is allowed. Please delete or rename the existing RayCluster before creating a new one with the desired name.
Response: {"kind":"Status","apiVersion":"v1","metadata":{},"status":"Failure","message":"rayclusters.ray.io \"raycluster-cpu\" already exists","reason":"AlreadyExists","details":{"name":"raycluster-cpu","group":"ray.io","kind":"rayclusters"},"code":409}

Waiting for requested resources to be set up...


KeyboardInterrupt: 

### Connect to a running cluster

If you've already created a cluster, but you've restarted the Python kernel, closed the notebook, or are working in a different notebook, and you want to connect to the existing cluster, uncomment the code in the following cell and then run it.

In [14]:
from codeflare_sdk import get_cluster
namespace = "fraud-detection"
name = "raycluster-cpu"
cluster = get_cluster(name, namespace=namespace)

Yaml resources loaded for raycluster-cpu


You can view information about the cluster, including a link to the Ray dashboard. In the Ray dashboard, you can inspect the running jobs and logs, and see the resources being used.
<figure>
    <img src="./assets/codeflare-details.png"  alt="codeflare details" width="400">
<figure/>



In [15]:
cluster.details()

                          🚀 CodeFlare Cluster Details 🚀                          
                                                                                   
 ╭───────────────────────────────────────────────────────────────────────────────╮ 
 │   Name                                                                        │ 
 │   raycluster-cpu                                                Inactive ❌   │ 
 │                                                                               │ 
 │   URI: ray://raycluster-cpu-head-svc.fraud-detection.svc:10001                │ 
 │                                                                               │ 
 │   ]8;id=231595;https://ray-dashboard-raycluster-cpu-fraud-detection.apps.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com\Dashboard🔗]8;;\                                                                 │ 
 │                                                                               │ 
 │                       Cluster Resources                                       │ 
 │   ╭── Workers ──╮  ╭───────── Worker specs(each) ─────────╮                   │ 
 │   │  # Workers  │  │  Memory      CPU         GPU         │                   │ 
 │   │             │  │                                      │                   │ 
 │   │  2          │  │  4G~2G       1~4         0           │                   │ 
 │   │             │  │                                      │                   │ 
 │   ╰─────────────╯  ╰──────────────────────────────────────╯                   │ 
 ╰───────────────────────────────────────────────────────────────────────────────╯

RayCluster(name='raycluster-cpu', status=<CodeFlareClusterStatus.STARTING: 2>, head_cpu_requests='2', head_cpu_limits='1', head_mem_requests='8G', head_mem_limits='5G', num_workers=2, worker_mem_requests='4G', worker_mem_limits='2G', worker_cpu_requests='1', worker_cpu_limits='4', namespace='fraud-detection', dashboard='https://ray-dashboard-raycluster-cpu-fraud-detection.apps.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com', worker_extended_resources={}, head_extended_resources={})

The link to the Ray dashboard is available in the cluster details provided as a result of running the previous cell.  It should look something like this:

<figure>
    <img src="./assets/ray-dashboard.png"  alt="ray dashboard" width="600"
<figure/>


## Ray job submission

### Initialize the Job Submission Client

If you want to submit jobs, connect to the running Ray cluster by initializing the job client that has the proper authentication and connection information.


In [19]:
import time
from requests.exceptions import HTTPError

# Initialize the job client with retry logic
max_retries = 5
retry_delay = 10  # seconds

for attempt in range(max_retries):
    try:
        print(f"Attempt {attempt + 1} to initialize job client...")
        client = cluster.job_client
        print("✓ Job client initialized successfully!")
        break
    except HTTPError as e:
        if attempt < max_retries - 1:
            print(f"✗ Failed to connect: {e}")
            print(f"Waiting {retry_delay} seconds before retry...")
            time.sleep(retry_delay)
        else:
            print(f"✗ Failed after {max_retries} attempts. The Ray cluster dashboard is not ready.")
            print("Please check the cluster status and try again later.")
            raise

Attempt 1 to initialize job client...
✓ Job client initialized successfully!


After you connect to the Ray cluster, you can query the cluster to determine whether there are any existing jobs:

In [28]:
client.list_jobs()

[]

### Create a runtime environment

Now you can configure the [runtime environment](https://docs.ray.io/en/latest/ray-core/handling-dependencies.html#runtime-environments) for the job. This step includes specifying the working directory, files to exclude, dependencies, and environment variables.

```python
runtime_env={
    "working_dir": "./", # relative path to files uploaded to the job
    "excludes": ["local_data/"], # directories and files to exclude from being uploaded to the job
    "pip": ["boto3", "botocore"], # can also be a string path to a requirements.txt file
    "env_vars": {
        "MY_ENV_VAR": "MY_ENV_VAR_VALUE",
        "MY_ENV_VAR_2": os.environ.get("MY_ENV_VAR_2"),
    },
}
```

In [29]:
import os

# script = "test_data_loader.py"
script = "train_tf_cpu.py"
runtime_env = {
    "working_dir": "./ray-scripts",
    "excludes": [],
    "pip": "./ray-scripts/requirements.txt",
    "env_vars": {
        "AWS_ACCESS_KEY_ID": os.environ.get("AWS_ACCESS_KEY_ID"),
        "AWS_SECRET_ACCESS_KEY": os.environ.get("AWS_SECRET_ACCESS_KEY"),
        "AWS_S3_ENDPOINT": os.environ.get("AWS_S3_ENDPOINT"),
        "AWS_DEFAULT_REGION": os.environ.get("AWS_DEFAULT_REGION"),
        "AWS_S3_BUCKET": os.environ.get("AWS_S3_BUCKET"),
        "NUM_WORKERS": "1",
        "TRAIN_DATA": "data/train.csv",
        "VALIDATE_DATA": "data/validate.csv",
        "MODEL_OUTPUT_PREFIX": "models/fraud/1/",
    },
}

### Submit the configured job

Now you can submit the job to the cluster. This step creates the necessary Kubernetes objects to run the job. The job runs the script with the specified runtime environment. The script for this example is located in [ray-scripts/train_tf_cpu.py](./ray-scripts/train_tf_cpu.py). The script follows the code fairly closely to the official [Ray TensorFlow example](https://docs.ray.io/en/latest/train/distributed-tensorflow-keras.html). This example uses TensorFlow, note that the [Ray site](https://docs.ray.io/en/latest/train/train.html) provides examples for PyTorch and other frameworks.

In [30]:
submission_id = client.submit_job(
    entrypoint=f"python {script}",
    runtime_env=runtime_env,
)

print(submission_id)

2026-03-25 02:47:04,673	INFO dashboard_sdk.py:355 -- Uploading package gcs://_ray_pkg_aebcf987b1e6c370.zip.
2026-03-25 02:47:04,674	INFO packaging.py:588 -- Creating a file package for local module './ray-scripts'.


raysubmit_FbBF7C2EGj6Qup6T


### Query important job information

In [31]:
# Get the job's status
print(client.get_job_status(submission_id), "\n")

# Get job related info
print(client.get_job_info(submission_id), "\n")

# Get the job's logs
print(client.get_job_logs(submission_id))

PENDING 

type=<JobType.SUBMISSION: 'SUBMISSION'> job_id=None submission_id='raysubmit_FbBF7C2EGj6Qup6T' driver_info=None status=<JobStatus.PENDING: 'PENDING'> entrypoint='python train_tf_cpu.py' message='Job has not started yet. It may be waiting for the runtime environment to be set up.' error_type=None start_time=1774406824718 end_time=None metadata={} runtime_env={'working_dir': 'gcs://_ray_pkg_aebcf987b1e6c370.zip', 'pip': {'packages': ['boto3~=1.35.12', 'botocore~=1.35.12', 'scikit-learn~=1.5.1', 'tensorflow~= 2.15.1', 'keras~=2.15.0', 'onnx~=1.16.2', 'tf2onnx~=1.16.1'], 'pip_check': False}, 'env_vars': {'AWS_ACCESS_KEY_ID': 'minio', 'AWS_SECRET_ACCESS_KEY': 'minio123', 'AWS_S3_ENDPOINT': 'https://minio-api-fraud-detection.apps.cluster-9xx9f.9xx9f.sandbox3611.opentlc.com', 'AWS_DEFAULT_REGION': 'local', 'AWS_S3_BUCKET': 'pipelines', 'NUM_WORKERS': '1', 'TRAIN_DATA': 'data/train.csv', 'VALIDATE_DATA': 'data/validate.csv', 'MODEL_OUTPUT_PREFIX': 'models/fraud/1/'}, '_ray_commit': '

You can also tail the job logs to watch the progress of the job.

In [32]:
# Iterate through the logs of a job 
async for lines in client.tail_job_logs(submission_id):
    print(lines, end="")

### List jobs

In [33]:
client.list_jobs()

[JobDetails(type=<JobType.SUBMISSION: 'SUBMISSION'>, job_id=None, submission_id='raysubmit_FbBF7C2EGj6Qup6T', driver_info=None, status=<JobStatus.FAILED: 'FAILED'>, entrypoint='python train_tf_cpu.py', message='runtime_env setup failed: Failed to set up runtime environment.\nCould not create the actor because its associated runtime env failed to be created.\n[Node 10.129.2.61] Traceback (most recent call last):\n  File "/opt/app-root/lib64/python3.12/site-packages/ray/_private/runtime_env/agent/runtime_env_agent.py", line 396, in _create_runtime_env_with_retry\n    runtime_env_context = await asyncio.wait_for(\n                          ^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/lib64/python3.12/asyncio/tasks.py", line 520, in wait_for\n    return await fut\n           ^^^^^^^^^\n  File "/opt/app-root/lib64/python3.12/site-packages/ray/_private/runtime_env/agent/runtime_env_agent.py", line 354, in _setup_runtime_env\n    await create_for_plugin_if_needed(\n  File "/opt/app-root/lib64/python

### Stop jobs

If you want to stop a job, call `stop_job` and specify the submission ID.  In the following cell, the command lists all the jobs and stops them.

In [26]:
for job_details in client.list_jobs():
    print(f"deleting {job_details.submission_id}")
    client.stop_job(job_details.submission_id)

deleting raysubmit_fqGQSAyb4qGNFf2h


### Delete jobs

You can also delete the jobs.

In [27]:
for job_details in client.list_jobs():
    print(f"deleting {job_details.submission_id}")
    client.delete_job(job_details.submission_id)

client.list_jobs()

deleting raysubmit_fqGQSAyb4qGNFf2h


[]

### Delete the cluster

After you complete training, you can delete the cluster. When you delete the cluster, you remove the Kubernetes objects and free up resources.

In [ ]:
cluster.down()